# Oracle ceiling control

Deliberate look-ahead, diagnostics only. Reproduces the figures the methodology
notes cite for the ceiling section, so a regenerated forecast cache cannot leave
them stale.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path("..").resolve()))

from backtest.targeting import TRADING_DAYS, BacktestConfig, build_ledger
from evaluation.forecasts import _qlike
from evaluation.oracle import daily_oracle, regime_oracle, sign_randomized_market
from evaluation.run_forecasts import DATASET_PATH, load_forecasts
from evaluation.strategy import evaluate_strategy, sharpe_ratio

TARGET_VOL = 0.10
pd.set_option("display.width", 250, "display.max_columns", 50)
pd.set_option("display.float_format", "{:.4f}".format)

In [2]:
df = pd.read_parquet(DATASET_PATH)
forecasts = load_forecasts()
regime = regime_oracle(df["realized_vol_gk"])
common = forecasts.index.intersection(regime.index)
realized = df.loc[common, "realized_vol_gk"]
vix_vol = df["vix"].shift(1) / 100.0 / np.sqrt(TRADING_DAYS)
config = BacktestConfig(execution_lag=1)
f"{common[0].date()} to {common[-1].date()}, {len(common)} days"

'2010-01-04 to 2026-06-01, 4127 days'

## Ceiling table

In [3]:
def strategy_metrics(name, vol):
    index = common.intersection(vol.dropna().index)
    ledger = build_ledger(df, vol.loc[index].rename("forecast"), config)
    ledger = ledger.dropna(subset=["net_return"])
    metrics = evaluate_strategy(
        ledger["net_return"], ledger["rf"], ledger["turnover"], TARGET_VOL
    )
    metrics["gross_sharpe"] = evaluate_strategy(
        ledger["gross_return"], ledger["rf"], ledger["turnover"], TARGET_VOL
    )["sharpe"]
    return metrics.rename(name)


columns = [
    strategy_metrics("oracle_daily", daily_oracle(realized)),
    strategy_metrics("oracle_regime", regime),
    strategy_metrics("oracle_regime_ahead", regime_oracle(df["realized_vol_gk"].shift(-1).dropna())),
    strategy_metrics("vix_sizing", vix_vol),
]
columns += [strategy_metrics(name, forecasts[name]) for name in forecasts.columns]

hold = df.loc[common]
buy_and_hold = evaluate_strategy(
    hold["ret"], hold["rf"], pd.Series(0.0, index=common), TARGET_VOL
).rename("buy_and_hold")
buy_and_hold["gross_sharpe"] = buy_and_hold["sharpe"]

table = pd.concat(columns + [buy_and_hold], axis=1).T
table[
    [
        "sharpe",
        "gross_sharpe",
        "realized_volatility",
        "vol_of_vol",
        "maximum_drawdown",
        "annualized_turnover",
        "worst_return_1",
    ]
]

,sharpe,gross_sharpe,realized_volatility,vol_of_vol,maximum_drawdown,annualized_turnover,worst_return_1
oracle_daily,2.0616,2.2097,0.0958,0.0161,-0.0783,70.6826,-0.0175
oracle_regime,1.8347,1.8422,0.0996,0.0212,-0.0855,3.8155,-0.0252
oracle_regime_ahead,1.8427,1.8500,0.1024,0.0239,-0.0807,3.8516,-0.0340
vix_sizing,0.8282,0.8400,0.0774,0.0331,-0.1114,4.5709,-0.0242
ewma,0.7824,0.7905,0.1039,0.0298,-0.1392,4.2025,-0.0470
garch,0.8006,0.8144,0.0979,0.0234,-0.1298,6.7250,-0.0402
har,0.8349,0.8661,0.0972,0.0223,-0.1555,15.1561,-0.0335
gbm,0.8086,0.8558,0.0986,0.0212,-0.1595,23.1746,-0.0385
gbm_vix,0.8015,0.8488,0.0974,0.0212,-0.1503,22.9277,-0.0343
trailing,0.8091,0.8161,0.1070,0.0317,-0.1637,3.7396,-0.0440


## Where the ceiling comes from

Sorting by future volatility recovers the drawdowns; sorting by anything known
at the decision date recovers the rebound instead, with the opposite sign.

In [4]:
def return_by_quintile(key):
    aligned = key.loc[common]
    return (
        df.loc[common, "ret"].groupby(pd.qcut(aligned, 5, labels=False)).mean() * 10_000
    ).round(1)


pd.DataFrame(
    {
        "contemporaneous_vol": return_by_quintile(realized),
        "regime_oracle": return_by_quintile(regime),
        "gbm_forecast": return_by_quintile(forecasts["gbm"]),
        "lagged_vix": return_by_quintile(vix_vol),
    }
).rename_axis("quintile")

,contemporaneous_vol,regime_oracle,gbm_forecast,lagged_vix
quintile,,,,
0,9.7000,15.2000,1.6000,3.1000
1,14.8000,15.1000,4.8000,5.1000
2,14.3000,8.9000,7.1000,1.8000
3,5.3000,2.8000,7.3000,8.9000
4,-14.3000,-12.3000,9.0000,10.8000


In [5]:
np.log(
    pd.DataFrame(
        {
            "regime_oracle": regime.loc[common],
            "vix_sizing": vix_vol.loc[common],
            "gbm": forecasts.loc[common, "gbm"],
            "trailing": forecasts.loc[common, "trailing"],
        }
    )
).corr()["regime_oracle"].round(3)

regime_oracle   1.0000
vix_sizing      0.6980
gbm             0.6640
trailing        0.6170
Name: regime_oracle, dtype: float64

## Timing versus risk control

Sign randomization keeps every absolute move, so the volatility path and every
forecast survive untouched, and breaks only the link between the size of a move
and its direction. What is left of an edge is risk control; what disappears is
return timing. The draws take a few minutes.

In [6]:
DRAWS = 100

sizers = {
    "oracle_daily": daily_oracle(realized),
    "oracle_regime": regime.loc[common],
    "vix_sizing": vix_vol.loc[common],
}
sizers |= {name: forecasts.loc[common, name] for name in forecasts.columns}


def sharpes(market):
    row = {}
    for name, vol in sizers.items():
        ledger = build_ledger(market, vol.rename("forecast"), config)
        ledger = ledger.dropna(subset=["net_return"])
        row[name] = sharpe_ratio(ledger["net_return"], ledger["rf"])
    hold = market.loc[common]
    row["buy_and_hold"] = sharpe_ratio(hold["ret"], hold["rf"])
    return pd.Series(row)


rng = np.random.default_rng(42)
actual = sharpes(df)
draws = pd.DataFrame(
    [sharpes(sign_randomized_market(df, common, rng)) for _ in range(DRAWS)]
)
excess = draws.sub(draws["buy_and_hold"], axis=0)

In [7]:
decomposition = pd.DataFrame(
    {
        "sharpe": actual,
        "excess": actual - actual["buy_and_hold"],
        "randomized_excess": excess.mean(),
        "p05": excess.quantile(0.05),
        "p95": excess.quantile(0.95),
    }
)
decomposition["timing"] = decomposition["excess"] - decomposition["randomized_excess"]
decomposition

,sharpe,excess,randomized_excess,p05,p95,timing
oracle_daily,2.0616,1.2734,0.3888,0.1817,0.6188,0.8846
oracle_regime,1.8347,1.0465,0.3039,0.0909,0.4969,0.7426
vix_sizing,0.8282,0.0400,0.2348,0.0916,0.3817,-0.1947
ewma,0.7824,-0.0058,0.2171,0.0154,0.4045,-0.2229
garch,0.8006,0.0124,0.2347,0.0408,0.4106,-0.2223
har,0.8349,0.0467,0.2538,0.0320,0.4532,-0.2071
gbm,0.8086,0.0205,0.2590,0.0424,0.4878,-0.2386
gbm_vix,0.8015,0.0134,0.2696,0.0635,0.4857,-0.2563
trailing,0.8091,0.0209,0.2284,0.0117,0.3908,-0.2075
buy_and_hold,0.7882,0.0000,0.0000,0.0000,0.0000,0.0000


The paired gap against the oracle, which is the statistic the ceiling section
cites, and the reason it has to be read paired: the same randomization moves
every strategy together, so the difference is far tighter than either level.

In [8]:
gap = draws["oracle_regime"] - draws["gbm"]
pd.Series(
    {
        "actual": actual["oracle_regime"] - actual["gbm"],
        "randomized": gap.mean(),
        "p05": gap.quantile(0.05),
        "p95": gap.quantile(0.95),
        "marginal_sd": draws["gbm"].std(),
        "paired_sd": gap.std(),
    }
)

actual         1.0260
randomized     0.0448
p05           -0.1403
p95            0.1797
marginal_sd    0.2523
paired_sd      0.0916
dtype: float64

## Where the VIX gain lands

In [9]:
losses = {
    name: _qlike(forecasts[name] ** 2, df.loc[forecasts.index, "realized_vol_gk"] ** 2)
    for name in ("gbm", "gbm_vix")
}
gain = losses["gbm"] - losses["gbm_vix"]
quintile = pd.qcut(df.loc[forecasts.index, "realized_vol_gk"], 5, labels=False)
(gain.groupby(quintile).sum() / gain.sum()).round(3).rename("share_of_qlike_gain")

realized_vol_gk
0   0.0890
1   0.0610
2   0.0350
3   0.1390
4   0.6760
Name: share_of_qlike_gain, dtype: float64